# 실습5 · 벡터 검색 — pgvector

실습1 에서 채운 임베딩으로 의미 검색을 한다. 원본 4096차원과 앞 128차원(MRL)을 비교하고, 128차원으로 후보를 추려 원본으로 다시 줄 세운다.
강의 : 「PGVector」 · 「ChromaDB vs PGVector」(고차원 재정렬용 / 저차원 인덱싱용 필드)

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결과 평가 도구 |
| 3 | 연산자 4개 — 잘라낸 벡터에서 무엇이 달라지나 |
| 4 | 전수 비교 vs HNSW 인덱스 |
| 5 | MRL — 몇 차원까지 잘라도 되나 |
| 6 | 후보 추리기 + 원본으로 재정렬 |
| 7 | 평가 종합 — BM25 · ts_rank 와 함께 |
| 8 | 서로 다른 질문을 맞힌다 |
| 9 | 내 질문으로 검색 (LM Studio) |

실습1 의 임베딩과 셀 10(HNSW 인덱스)이 끝나 있어야 한다. 셀 9 만 LM Studio 서버가 필요하다.

### 셀 1 · 설정
- 셀 9 에서 LM Studio 로 내 질문을 임베딩한다. 주소·모델은 실습1 셀 1 과 같게

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
EVAL_N = 500
QID    = 1                                             # 셀 3 에서 볼 질문 id

LMSTUDIO_URL = "http://host.docker.internal:1234/v1"    # 셀 9 에서만 사용
EMBED_MODEL  = "text-embedding-qwen3-embedding-8b"
QUERY_TASK   = "Given a Korean question, retrieve the news or Wikipedia passage that answers the question"
MY_QUERY     = "전기차 배터리 화재를 막는 기술"            # 셀 9 에서 검색해 볼 내 질문

### 셀 2 · 연결과 평가 도구

In [ ]:
# 셀 2 · 연결과 평가 도구
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DB_URL, autocommit=True)
conn.prepare_threshold = None      # SET 으로 바꾼 설정이 매번 반영되게 (실행 계획 재사용 끔)
register_vector(conn)              # numpy 배열 ↔ vector 타입


def sql(query, params=None):
    """SELECT 결과를 표(DataFrame)로"""
    cur = conn.execute(query, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]) if cur.description else None


# 평가용 질문 : 11개마다 하나씩 고른 EVAL_N 개. 모든 노트북이 같은 질문으로 잰다
E = f"(SELECT id, doc_id AS gold, question, embedding, emb128 FROM questions WHERE id % 11 = 0 ORDER BY id LIMIT {EVAL_N}) e"
RESULTS = Path("results/검색평가.csv")


def evaluate(name, query, params=None):
    """query 는 (질문 id, 정답 지문 id, 상위 문서 id 배열) 을 질문마다 한 행씩 돌려주는 SQL.
    정답 지문이 상위 10위 안이면 Hit, MRR = 정답 순위 역수의 평균. 결과는 RESULTS 에 누적"""
    print(f"{name} …", end=" ", flush=True)          # 방법 하나가 끝날 때마다 한 줄씩 보이게
    t = time.perf_counter()
    rows = conn.execute(query, params).fetchall()
    sec = time.perf_counter() - t
    rank = [list(ids[:10]).index(g) + 1 if g in ids[:10] else None for _, g, ids in rows]
    res = {"방법": name, "Hit@10": round(np.mean([r is not None for r in rank]), 3),
           "MRR": round(np.mean([1 / r if r else 0 for r in rank]), 3), "질문수": len(rows), "초": round(sec, 1)}
    RESULTS.parent.mkdir(exist_ok=True)
    new = pd.DataFrame([res])
    if RESULTS.exists():
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old["방법"] != name], new])
    new.to_csv(RESULTS, index=False)
    print(f"Hit@10 {res['Hit@10']:.3f} · MRR {res['MRR']:.3f} · {res['초']}s", flush=True)
    return res


n = conn.execute("SELECT count(*), count(embedding) FROM docs").fetchone()
assert n[0] == n[1], "실습1 의 지문 임베딩이 다 차야 한다"
print("지문", n[0], "개 임베딩 완료 · 평가 질문", conn.execute(f"SELECT count(*) FROM {E}").fetchone()[0], "개")

### 셀 3 · 연산자 4개
질문 하나와 정답 지문 하나의 거리. 세 벌의 벡터로 잰다.
- 원본 4096 (길이 1) · 앞 128 을 **자르기만** 한 것(길이 약 0.2) · emb128 = 잘라서 다시 길이 1
- `<->` L2 거리 · `<#>` 음의 내적 · `<=>` 코사인 거리 · `<+>` L1 거리
- 코사인은 길이를 나눠 없애므로 자르기만 해도 같다. 내적·L2 는 길이에 따라 달라진다

In [ ]:
# 셀 3 · 연산자 4개
sql("""
WITH p AS (SELECT q.embedding AS qe, d.embedding AS de FROM questions q JOIN docs d ON d.id = q.doc_id WHERE q.id = %(qid)s),
v AS (SELECT '원본 4096' AS 벡터, qe AS a, de AS b FROM p
      UNION ALL SELECT '앞 128 자르기만', subvector(qe, 1, 128), subvector(de, 1, 128) FROM p
      UNION ALL SELECT 'emb128 (다시 길이 1)', l2_normalize(subvector(qe, 1, 128)), l2_normalize(subvector(de, 1, 128)) FROM p)
SELECT 벡터, round(vector_norm(a)::numeric, 4) AS 질문길이, round(vector_norm(b)::numeric, 4) AS 지문길이,
       round((a <-> b)::numeric, 4) AS "<-> L2",  round((a <#> b)::numeric, 4) AS "<#> 음의내적",
       round((a <=> b)::numeric, 4) AS "<=> 코사인거리", round((a <+> b)::numeric, 4) AS "<+> L1"
FROM v
""", {"qid": QID})

### 셀 4 · 전수 비교 vs HNSW 인덱스
- 원본 4096 : 인덱스 없음 → 지문 5,309개와 모두 비교(Seq Scan + Sort)
- emb128 : `docs_emb128_hnsw` 인덱스로 일부만 비교(Index Scan)

In [ ]:
# 셀 4 · 실행 계획과 시간
for col in ["embedding", "emb128"]:
    print(f"── {col}")
    for (line,) in conn.execute(f"""EXPLAIN (ANALYZE, COSTS OFF)
            SELECT d.id FROM docs d ORDER BY d.{col} <=> (SELECT {col} FROM questions WHERE id = %s) LIMIT 10""", (QID,)):
        print("  ", line)
t = {}
for col in ["embedding", "emb128"]:
    s = time.perf_counter()
    for qid in range(1, 51):
        conn.execute(f"SELECT d.id FROM docs d ORDER BY d.{col} <=> (SELECT {col} FROM questions WHERE id = %s) LIMIT 10", (qid,)).fetchall()
    t[col] = (time.perf_counter() - s) / 50 * 1000
print(f"\n질의당 평균 : 원본 4096 전수 {t['embedding']:.1f} ms · emb128 HNSW {t['emb128']:.1f} ms")

### 셀 5 · MRL — 몇 차원까지 잘라도 되나
원본 벡터를 파이썬으로 가져와 앞 d 차원만 잘라 다시 길이 1로 → 전수 비교로 Hit@10 · MRR. (인덱스 없이 정확 검색)
- 오른쪽 축 : 벡터 1개의 저장 크기(float32 4바이트 × d)

In [ ]:
# 셀 5 · MRL 차원 스윕 (numpy 전수 비교)
ids, X = zip(*conn.execute("SELECT id, embedding FROM docs ORDER BY id").fetchall())
ids, X = np.array(ids), np.stack(X).astype(np.float32)
Q = sql(f"SELECT e.id, e.gold, e.embedding FROM {E}")
QX = np.stack(Q["embedding"]).astype(np.float32)
rows = []
for d in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    Xd = X[:, :d] / np.linalg.norm(X[:, :d], axis=1, keepdims=True)
    Qd = QX[:, :d] / np.linalg.norm(QX[:, :d], axis=1, keepdims=True)
    top = ids[np.argsort(-(Qd @ Xd.T), axis=1)[:, :10]]
    r = [list(t).index(g) + 1 if g in t else None for t, g in zip(top, Q["gold"])]
    rows.append({"차원": d, "Hit@10": np.mean([x is not None for x in r]), "MRR": np.mean([1 / x if x else 0 for x in r]),
                 "벡터 크기(KB)": d * 4 / 1024})
M = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(M["차원"], M["Hit@10"], "-o", color="#3D484A", label="Hit@10")
ax.plot(M["차원"], M["MRR"], "-o", color="#FF7043", label="MRR")
ax.set_xscale("log", base=2); ax.set_xticks(M["차원"], M["차원"]); ax.set_ylim(0, 1); ax.set_xlabel("앞에서 자른 차원 수")
ax.legend(); ax.set_title("MRL : 앞 d 차원만 써서 전수 비교"); plt.show()
M.round(3)

### 셀 6 · 후보 추리기 + 원본으로 재정렬
emb128 HNSW 로 후보 N개 → 그 N개만 원본 4096 으로 다시 줄 세워 상위 10.
- `hnsw.ef_search` : HNSW 가 넓게 보는 후보 폭(기본 40). **결과 행 수도 이 값을 넘지 못함** → N 이 40보다 크면 같이 올린다

In [ ]:
# 셀 6 · 후보 N개 → 원본 재정렬
def rerank_sql(n):
    return f"""SELECT e.id, e.gold, ARRAY(
                 SELECT c.id FROM (SELECT d.id, d.embedding FROM docs d ORDER BY d.emb128 <=> e.emb128 LIMIT {n}) c
                 ORDER BY c.embedding <=> e.embedding LIMIT 10) FROM {E}"""

print("ef_search 40 에서 LIMIT 100 을 요청하면 :",
      conn.execute("SELECT count(*) FROM (SELECT id FROM docs ORDER BY emb128 <=> (SELECT emb128 FROM questions WHERE id = 1) LIMIT 100) x").fetchone()[0], "행")
rows = []
for n in [10, 20, 40, 80, 160]:
    conn.execute(f"SET hnsw.ef_search = {max(40, n)}")
    rows.append({"후보 N": n, "ef_search": max(40, n), **evaluate(f"emb128 후보 {n} → 4096 재정렬", rerank_sql(n))})
conn.execute("RESET hnsw.ef_search")
C = pd.DataFrame(rows)
C[["후보 N", "ef_search", "Hit@10", "MRR", "초"]]

### 셀 7 · 평가 종합
원본 4096 전수 비교와 emb128 HNSW 단독도 평가해 BM25 · ts_rank 와 한 표에 모은다. 약 1분.

In [ ]:
# 셀 7 · 평가 종합
evaluate("벡터 4096 전수", f"SELECT e.id, e.gold, ARRAY(SELECT d.id FROM docs d ORDER BY d.embedding <=> e.embedding LIMIT 10) FROM {E}")
evaluate("벡터 emb128 HNSW", f"SELECT e.id, e.gold, ARRAY(SELECT d.id FROM docs d ORDER BY d.emb128 <=> e.emb128 LIMIT 10) FROM {E}")
R = pd.read_csv(RESULTS)
R = R[~R["방법"].str.startswith("BM25 k1=") & ~R["방법"].str.match(r"emb128 후보 (?:10|20|80|160) ")].sort_values("MRR", ascending=False)
fig, ax = plt.subplots(figsize=(12, 0.45 * len(R) + 1))
y = np.arange(len(R))
ax.barh(y + 0.2, R["Hit@10"], 0.4, color="#3D484A", label="Hit@10"); ax.barh(y - 0.2, R["MRR"], 0.4, color="#FF7043", label="MRR")
for i, (h, m) in enumerate(R[["Hit@10", "MRR"]].values):
    ax.text(h, i + 0.2, f" {h:.3f}", va="center", fontsize=8); ax.text(m, i - 0.2, f" {m:.3f}", va="center", fontsize=8)
ax.set_yticks(y, R["방법"]); ax.invert_yaxis(); ax.set_xlim(0, 1.1); ax.legend(loc="lower right"); plt.show()
R

### 셀 8 · 서로 다른 질문을 맞힌다
BM25 와 벡터(4096 전수)가 각각 맞힌 질문을 교차로 센다. 한쪽만 맞힌 질문이 있다면 둘을 합칠 이유가 된다(4-2 하이브리드).

In [ ]:
# 셀 8 · BM25 vs 벡터 교차표
rows = conn.execute(f"""SELECT e.id, e.gold, e.question,
        ARRAY(SELECT doc_id FROM bm25_search(e.question, 10)),
        ARRAY(SELECT d.id FROM docs d ORDER BY d.embedding <=> e.embedding LIMIT 10) FROM {E}""").fetchall()
X8 = pd.DataFrame([{"질문": q, "BM25": g in b, "벡터": g in v,
                    "BM25순위": b.index(g) + 1 if g in b else None, "벡터순위": v.index(g) + 1 if g in v else None}
                   for _, g, q, b, v in rows])
display(pd.crosstab(X8["BM25"].map({True: "BM25 맞힘", False: "BM25 놓침"}), X8["벡터"].map({True: "벡터 맞힘", False: "벡터 놓침"})))
display(X8[X8["벡터"] & ~X8["BM25"]].head(5)[["질문", "벡터순위"]])
X8[X8["BM25"] & ~X8["벡터"]].head(5)[["질문", "BM25순위"]]

### 셀 9 · 내 질문으로 검색 (LM Studio)
`MY_QUERY` 를 질문과 같은 형식(지시문 + Query:)으로 임베딩 → emb128 후보 40 → 4096 재정렬. BM25 결과와 나란히.
먼저 서버와 모델 이름을 확인한다. 서버가 꺼져 있거나 주소 · 모델 이름이 다르면 이 셀만 멈추고, 셀 1 에서 무엇을 고칠지 알려 준다.

In [ ]:
# 셀 9 · 내 질문으로 검색
import requests
try:
    models = [m["id"] for m in requests.get(f"{LMSTUDIO_URL}/models", timeout=5).json()["data"]]
except requests.exceptions.RequestException:
    raise ConnectionError(f"LM Studio 에 연결할 수 없음 : {LMSTUDIO_URL}\n"
                          "→ LM Studio 서버를 켜고, Developer 탭에 보이는 포트로 셀 1 의 LMSTUDIO_URL 을 고친 뒤 셀 1 부터 다시 실행") from None
assert EMBED_MODEL in models, f"셀 1 의 EMBED_MODEL 을 이 목록의 이름으로 바꾼다 : {models}"

r = requests.post(f"{LMSTUDIO_URL}/embeddings", timeout=120,
                  json={"model": EMBED_MODEL, "input": [f"Instruct: {QUERY_TASK}\nQuery:{MY_QUERY}"]})
r.raise_for_status()
v = np.array(r.json()["data"][0]["embedding"], dtype=np.float32)
vec = sql("""SELECT c.id, round((1 - (c.embedding <=> %(v)s))::numeric, 4) AS 유사도, left(c.title, 40) AS 제목
             FROM (SELECT id, title, embedding FROM docs
                   ORDER BY emb128 <=> l2_normalize(subvector(%(v)s, 1, 128))::vector(128) LIMIT 40) c
             ORDER BY c.embedding <=> %(v)s LIMIT 5""", {"v": v})
bm = sql("SELECT r.doc_id AS id, round(r.score::numeric, 2) AS bm25, left(d.title, 40) AS 제목 "
         "FROM bm25_search(%s, 5) r JOIN docs d ON d.id = r.doc_id ORDER BY r.score DESC", (MY_QUERY,))
pd.concat({"벡터": vec, "BM25": bm}, axis=1)